# 8 — LLM-Powered Idea Generation
## Parse the Web, Score Ideas — A Two-Stage LLM Pipeline
⏱ ~12 min

**Idea generation** is one of the most compelling creative applications of LLMs. Instead of generating raw text, this workshop builds a *structured* two-agent pipeline: one agent parses and organises a tool catalogue, and a second agent proposes novel product ideas that combine those tools. The key insight is that **separating structure from generation** produces higher-quality, more reliable output than prompting a single model to do both.

By the end of this session you will understand how to:
- Model complex agent state with nested Pydantic objects inside `TypedDict`
- Build a two-node LangGraph pipeline with explicit typed contracts between nodes
- Extract and validate JSON reliably from LLM responses (fences, fallbacks, schema enforcement)
- Score and rank generated ideas by effort and value
- Extend the pipeline with a third node (scorer, filter, or exporter)

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why separate parsing from generation? |
| 2 | **Data Models** — TypedDict state + Pydantic contracts |
| 3 | **Parser Node** — Structuring raw data into typed form |
| 4 | **Generator Node** — JSON extraction and idea modelling |
| 5 | **Graph Assembly** — Wiring nodes with LangGraph |
| 6 | **Scoring Extension** — Adding a third pipeline node |
| 7 | **Debugging** — Inspecting state at each step |
| ★ | **Advanced** — Fetching live data + conditional routing |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing the project requirements (or run on Google Colab)
- An `OPENAI_API_KEY` in your `.env` file or Colab Secrets

### Key References
> Brown, T., Mann, B., et al. (2020). *Language Models are Few-Shot Learners.* NeurIPS 2020. https://arxiv.org/abs/2005.14165
>
> Wei, J., Wang, X., et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models.* NeurIPS 2022. https://arxiv.org/abs/2201.11903
>
> LangGraph documentation: https://langchain-ai.github.io/langgraph/
>
> free-for-dev list (the domain used in this example): https://github.com/ripienaar/free-for-dev

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "pydantic",
            "python-dotenv",
            "requests",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY not found or looks wrong. "
    "Local: add it to .env. Colab: add it in the Secrets panel."
)
print(f"OPENAI_API_KEY present: {key[:8]}...")

---

## Part 1 — Why Separate Parsing from Generation?

---

### The Single-Prompt Trap

A naive approach to LLM-assisted ideation dumps everything into one prompt:

```
"Here is a big list of tools. Generate product ideas."
```

This works for toy demos but breaks in production because:

1. **Data quality is unpredictable** — the LLM may misread or hallucinate tool names from messy input.
2. **No schema enforcement** — output format drifts across calls, making downstream parsing fragile.
3. **Hard to debug** — when output is wrong you cannot tell if the problem is in parsing, reasoning, or generation.
4. **No reuse** — you cannot swap the generator without rewriting the parser.

### The Two-Agent Pattern

The solution is to decompose the task into two focused agents with an explicit typed contract between them:

| Stage | Responsibility | Output type |
|-------|---------------|-------------|
| **Parser agent** | Extract structure from raw input | `ParsedContent` (Pydantic) |
| **Generator agent** | Create ideas from structured data | `list[IdeaOutput]` (Pydantic) |

Each agent does exactly one thing. The Pydantic models are the contract — if the parser produces valid `ParsedContent`, the generator is guaranteed to receive a clean, typed object it can iterate over without defensive checks.

### Brainstorming Approach Comparison

| Approach | Quality | Reliability | Debuggability | Reusability |
|----------|---------|-------------|---------------|-------------|
| **Single prompt** (dump-everything) | Medium | Low | Poor | None |
| **Chain-of-Thought** (one prompt, structured thinking) | High | Medium | Medium | Low |
| **Two-agent pipeline** (this workshop) | High | High | Excellent | High |
| **Multi-agent with critic** | Very high | High | Excellent | High |

> Wei et al. (2022) showed that prompting models to reason step-by-step before producing answers dramatically improves quality on complex tasks. The two-agent pipeline extends this: each agent *is* a reasoning step, with a typed state object capturing the intermediate result rather than prose.

### LLM-Assisted Ideation Patterns

| Pattern | Description | Best for |
|---------|-------------|----------|
| **Catalogue + Combine** | Parse a catalogue of primitives, ask LLM to combine them | Tool/API mashup ideas (this workshop) |
| **Problem-first** | Define a pain point, ask LLM for solutions | User research → product exploration |
| **Constraint-driven** | Fix budget/time/stack, generate within constraints | Hackathon or MVP planning |
| **Analogy transfer** | Describe a pattern from one domain, apply it to another | Cross-industry innovation |
| **Scoring loop** | Generate N ideas, score each, keep top-k | Quality filtering at scale |

### Pipeline Architecture

```
SETUP PHASE (run once per domain)
─────────────────────────────────────────────────────

  Raw input (tool catalogue, markdown, API response)
       │
       ▼
  ┌────────────────┐
  │  Parser Node   │  extracts names, categories,
  │                │  descriptions → validates with
  └───────┬────────┘  Pydantic (FreeTool, ParsedContent)
           │
           ▼
  state["parsed_content"]   ← typed contract
       (ParsedContent)


GENERATION PHASE (run per ideation request)
─────────────────────────────────────────────────────

  state["parsed_content"]
       │
       ▼
  ┌────────────────┐
  │ Generator Node │  formats tool list into prompt,
  │                │  calls LLM, strips JSON fences,
  └───────┬────────┘  validates with Pydantic
           │
           ▼
  state["ideas"]
  list[IdeaOutput]  ← typed results
       │
       ▼
  ┌────────────────┐
  │  Scorer Node   │  ranks ideas by effort + value
  │  (Part 6)      │  returns state["scored_ideas"]
  └───────┬────────┘
           │
           ▼
        END  →  display / save to JSON
```

> **Key design choice:** the generator node reads `state["parsed_content"]` but has no idea how it was populated. You could swap the parser node for one that fetches from a database, API, or file — the generator stays unchanged. This is the power of typed state as a contract.

### State Shape

```
AgentState
├── parsed_content: ParsedContent | None
│   ├── tools: list[FreeTool]
│   │   ├── name: str
│   │   ├── category: str
│   │   └── description: str
│   ├── categories: list[str]
│   └── summary: str
└── ideas: list[IdeaOutput] | None
    ├── idea: IdeaConcept
    │   ├── title, description
    │   ├── target_tools: list[str]
    │   ├── value_proposition: str
    │   └── effort_level: str
    ├── reasoning: str
    └── next_steps: list[str]
```

---

## Part 2 — Data Models: TypedDict State + Pydantic Contracts

---

### Why Two Model Systems?

LangGraph uses `TypedDict` for graph state because it needs a simple, mutable dict-like container that can be partially updated by each node. Pydantic models live *inside* the state dict as field values — they provide validation, serialisation, and IDE autocompletion for the actual domain objects.

```
TypedDict  →  controls what keys the graph state has
Pydantic   →  validates the objects stored at each key
```

### Model Hierarchy

| Model | Role | Lives in |
|-------|------|----------|
| `FreeTool` | One entry from the tool catalogue | `ParsedContent.tools` |
| `ParsedContent` | Full structured catalogue | `AgentState["parsed_content"]` |
| `IdeaConcept` | The core of one idea | `IdeaOutput.idea` |
| `IdeaOutput` | Idea + reasoning + next steps | `AgentState["ideas"][i]` |
| `AgentState` | Top-level graph state | LangGraph `StateGraph` |

In [ ]:
# ─── 2-A: Define all data models ──────────────────────────────────────────────
# These models form the typed contract between pipeline nodes.
# The parser fills parsed_content; the generator reads it to produce ideas.
from typing import TypedDict
from pydantic import BaseModel, Field


class FreeTool(BaseModel):
    """One free developer tool extracted from the catalogue."""
    name: str
    category: str
    description: str


class ParsedContent(BaseModel):
    """Structured output of the parser node."""
    tools: list[FreeTool]
    categories: list[str]
    summary: str


class IdeaConcept(BaseModel):
    """The core concept of one generated idea."""
    title: str
    description: str
    target_tools: list[str] = Field(description="Names of 2-3 tools this idea combines")
    value_proposition: str
    effort_level: str = Field(description="One of: low, medium, high")


class IdeaOutput(BaseModel):
    """Full output for one idea including reasoning and action plan."""
    idea: IdeaConcept
    reasoning: str
    next_steps: list[str]


class AgentState(TypedDict):
    """Top-level graph state. Each node writes to a subset of these keys."""
    parsed_content: ParsedContent | None
    ideas: list[IdeaOutput] | None


print("Models defined.")
print("Flow: AgentState[parsed_content] → parser → AgentState[ideas] → generator")

In [ ]:
# ─── 2-B: Inspect the model schemas ──────────────────────────────────────────
# Pydantic v2 generates a JSON schema for every model.
# This is the exact schema the LLM must satisfy when returning ideas.
import json

print("=== IdeaConcept schema (what the LLM must produce) ===")
print(json.dumps(IdeaConcept.model_json_schema(), indent=2))
print()
print("=== FreeTool schema ===")
print(json.dumps(FreeTool.model_json_schema(), indent=2))

---

## Part 3 — Parser Node: Structuring Raw Data into Typed Form

---

### What the Parser Does

The parser's only job is to take raw input and produce a validated `ParsedContent` object. It:

1. Instantiates `FreeTool` for each entry — Pydantic validates field types immediately
2. Derives the category list by de-duplicating tool categories
3. Writes a human-readable summary string for logging
4. Returns `{"parsed_content": <ParsedContent>}` — LangGraph merges this into state

### Why Not Call the LLM in the Parser?

In this example the data is already structured, so no LLM call is needed. In the production `src/tools.py`, a regex-based Markdown parser extracts tools from a GitHub README. The rule is: **use the LLM only when rule-based parsing is insufficient**. LLM calls add latency and cost; deterministic parsing is faster, cheaper, and fully reproducible.

### Tool Catalogue

The tools below are a curated sample from [free-for-dev](https://github.com/ripienaar/free-for-dev). In the production script (`src/tools.py`), the full list of 300+ tools is fetched live from GitHub at runtime.

In [ ]:
# ─── 3-A: Define the tool catalogue ──────────────────────────────────────────
# Production: fetched from https://github.com/ripienaar/free-for-dev
# Workshop:   inline list — no HTTP request needed

FREE_TOOLS_RAW = [
    {"name": "Google Colab",       "category": "Machine Learning",  "description": "Free cloud Jupyter notebooks with GPU access"},
    {"name": "Hugging Face",        "category": "Machine Learning",  "description": "Model hub with free inference API tier"},
    {"name": "GitHub Actions",      "category": "CI/CD",             "description": "Automated workflows up to 2000 min/month free"},
    {"name": "Vercel",              "category": "Hosting",           "description": "Free tier for frontend deployments with previews"},
    {"name": "Supabase",            "category": "Database",          "description": "Postgres + auth + storage with free tier"},
    {"name": "Cloudflare Workers",  "category": "Serverless",        "description": "100k free requests/day, runs at the edge"},
    {"name": "Upstash",             "category": "Database",          "description": "Serverless Redis, free tier with 10k req/day"},
    {"name": "Resend",              "category": "Email",             "description": "100 emails/day free, developer-friendly API"},
    {"name": "Clerk",               "category": "Authentication",    "description": "User auth with social login, free up to 5k MAU"},
    {"name": "Sentry",              "category": "Error Tracking",    "description": "Error monitoring, free 5k events/month"},
    {"name": "PostHog",             "category": "Analytics",         "description": "Product analytics, free up to 1M events/month"},
    {"name": "Trigger.dev",         "category": "Background Jobs",   "description": "Background job queue, free tier available"},
    {"name": "Render",              "category": "Hosting",           "description": "Free tier for web services (sleep on idle)"},
    {"name": "Cal.com",             "category": "Scheduling",        "description": "Open-source scheduling, free self-host or cloud"},
    {"name": "Liveblocks",          "category": "Collaboration",     "description": "Real-time collaboration infra, free 100 MAU"},
]

print(f"Catalogue: {len(FREE_TOOLS_RAW)} tools")
categories = sorted(set(t["category"] for t in FREE_TOOLS_RAW))
print(f"Categories ({len(categories)}): {', '.join(categories)}")

In [ ]:
# ─── 3-B: Parser node ─────────────────────────────────────────────────────────
# LangGraph calls this function with the current AgentState dict.
# It returns a dict with only the keys it wants to update.

def parser_node(state: AgentState) -> dict:
    """Convert raw tool data into a validated ParsedContent object."""
    tools = [FreeTool(**t) for t in FREE_TOOLS_RAW]
    cats = sorted(set(t.category for t in tools))
    content = ParsedContent(
        tools=tools,
        categories=cats,
        summary=(
            f"Found {len(tools)} free tools across {len(cats)} categories: "
            f"{', '.join(cats)}"
        ),
    )
    print(f"  [parser] structured {len(tools)} tools in {len(cats)} categories")
    return {"parsed_content": content}


# Test it standalone — always test nodes before wiring into the graph
initial_state: AgentState = {"parsed_content": None, "ideas": None}
parser_result = parser_node(initial_state)
parsed = parser_result["parsed_content"]

print(f"\nSummary: {parsed.summary}")
print(f"First tool: {parsed.tools[0]}")
print(f"Type: {type(parsed.tools[0]).__name__} — Pydantic validated")

In [ ]:
# ─── 3-C: Browse the catalogue by category ───────────────────────────────────
# Useful sanity check before running generation — confirm categories look right.

from collections import defaultdict

by_category = defaultdict(list)
for tool in parsed.tools:
    by_category[tool.category].append(tool.name)

print("Tools by category:\n")
for cat in sorted(by_category):
    names = ", ".join(by_category[cat])
    print(f"  {cat:20} {names}")

---

## Part 4 — Generator Node: JSON Extraction and Idea Modelling

---

### The JSON Extraction Problem

LLMs often wrap JSON in markdown code fences even when you ask for raw JSON:

```
Sure! Here are some ideas:

```json
[{"title": "...", ...}]
```
```

A robust extraction strategy handles three cases:

```
Case 1 — clean JSON:       [{...}]
Case 2 — fenced JSON:      ```json\n[{...}]\n```
Case 3 — prose + JSON:     "Here are my ideas: [{...}]"
```

The pattern used in this pipeline:
1. Try to match a ` ```json ... ``` ` fence — extract the inner content
2. If no fence, search for the first `[` ... `]` block
3. If both fail, return an empty list rather than raising an exception

### Prompt Engineering for Structured Output

| Technique | Effect |
|-----------|--------|
| Include the JSON schema in the system prompt | Reduces field omissions and naming drift |
| Ask for a JSON array at the top level | Simpler to parse than `{"ideas": [...]}` wrappers |
| Specify `effort_level` as an explicit enum | Prevents freeform values like "moderate" |
| Set `temperature=0.7` | Balances creativity with consistency |
| Ask for `next_steps` as an array of 3 strings | Forces concrete, iterable action items |

> Brown et al. (2020) demonstrated that few-shot examples in the prompt dramatically improve the structure of LLM output. Adding one well-formed example idea to the system prompt is often more effective than detailed prose instructions.

In [ ]:
# ─── 4-A: JSON extraction utility ────────────────────────────────────────────
# This is the most brittle part of any LLM pipeline — test it thoroughly.
import re


def extract_json_array(raw: str) -> list:
    """Robustly extract a JSON array from LLM output.

    Handles three forms:
      1. Bare JSON array:        [{...}]
      2. Fenced JSON:            ```json\n[{...}]\n```
      3. Prose-wrapped JSON:     'Here are ideas: [{...}]'

    Returns an empty list if nothing parses cleanly.
    """
    text = raw.strip()

    # Case 2: strip ```json ... ``` fence
    fence = re.search(r"```(?:json)?\s*(\[.*?\])\s*```", text, re.DOTALL)
    if fence:
        text = fence.group(1)
    else:
        # Case 3: find the first complete [...] block in any surrounding prose
        arr_match = re.search(r"(?s)(\[.*\])", text)
        if arr_match:
            text = arr_match.group(1)

    try:
        data = json.loads(text)
        return data if isinstance(data, list) else []
    except json.JSONDecodeError:
        return []


# Test all three cases
test_cases = [
    ('[{"title": "Idea A", "effort_level": "low"}]',
     "Case 1: clean array"),
    ('```json\n[{"title": "Idea B", "effort_level": "medium"}]\n```',
     "Case 2: fenced"),
    ('Sure! Here are ideas:\n[{"title": "Idea C", "effort_level": "high"}]',
     "Case 3: prose-wrapped"),
    ('Sorry, I cannot output JSON today.',
     "Case 4: broken → returns []"),
]

for raw, label in test_cases:
    result = extract_json_array(raw)
    print(f"{label}")
    print(f"  → {result}")

In [ ]:
# ─── 4-B: Generator node ─────────────────────────────────────────────────────
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

_SYSTEM_PROMPT = """
You are a product ideation expert. Given a list of free developer tools, generate 3-5
practical product ideas that each combine exactly 2-3 tools.

Return a JSON array only — no prose, no markdown fences. Each object must have:
  title             (string)
  description       (string, 1-2 sentences)
  target_tools      (array of 2-3 tool name strings, must match catalogue names exactly)
  value_proposition (string, what problem this solves)
  effort_level      (string, exactly one of: low / medium / high)
  reasoning         (string, why these tools work together)
  next_steps        (array of exactly 3 strings, concrete implementation steps)
""".strip()


def generator_node(state: AgentState) -> dict:
    """Generate product ideas from the parsed tool catalogue."""
    content = state["parsed_content"]
    tool_list = "\n".join(
        f"- {t.name} ({t.category}): {t.description}" for t in content.tools
    )

    print(f"  [generator] calling LLM with {len(content.tools)} tools...")

    resp = _llm.invoke(
        [
            SystemMessage(_SYSTEM_PROMPT),
            HumanMessage(f"Available free tools:\n{tool_list}\n\nGenerate 3-5 ideas:"),
        ]
    )

    ideas_data = extract_json_array(resp.content)

    if not ideas_data:
        print("  [generator] WARNING: JSON extraction failed — returning empty list")
        print(f"  Raw response: {resp.content[:200]}")
        return {"ideas": []}

    ideas = [
        IdeaOutput(
            idea=IdeaConcept(
                title=d["title"],
                description=d.get("description", ""),
                target_tools=d.get("target_tools", []),
                value_proposition=d.get("value_proposition", ""),
                effort_level=d.get("effort_level", "medium"),
            ),
            reasoning=d.get("reasoning", ""),
            next_steps=d.get("next_steps", []),
        )
        for d in ideas_data
        if isinstance(d, dict) and "title" in d
    ]

    print(f"  [generator] produced {len(ideas)} validated ideas")
    return {"ideas": ideas}


print("Generator node ready.")

In [ ]:
# ─── 4-C: Test the generator node in isolation ───────────────────────────────
# Always test nodes independently before wiring them into a graph.
# We inject the parser output directly so this test is self-contained.

test_state: AgentState = {
    "parsed_content": parsed,   # from Part 3
    "ideas": None,
}

gen_result = generator_node(test_state)

print(f"\n=== GENERATOR OUTPUT: {len(gen_result['ideas'])} ideas ===")
for i, item in enumerate(gen_result["ideas"], 1):
    idea = item.idea
    print(f"\n{i}. {idea.title} [{idea.effort_level} effort]")
    print(f"   {idea.description}")
    print(f"   Tools: {', '.join(idea.target_tools)}")
    print(f"   Value: {idea.value_proposition}")

---

## Part 5 — Graph Assembly: Wiring Nodes with LangGraph

---

### How LangGraph Manages State

LangGraph `StateGraph` works like a directed graph where:
- **Nodes** are Python functions that receive the full state and return a partial update dict
- **Edges** define execution order (or conditional routing)
- **State** is merged automatically — each node only needs to return the keys it changes

```
START
  │
  ▼
parser_node(state) → {"parsed_content": <ParsedContent>}
  │                   ↑ merged into state automatically
  ▼
generator_node(state) → {"ideas": [<IdeaOutput>, ...]}
  │                       ↑ merged into state automatically
  ▼
END
```

### LCEL vs LangGraph

| Dimension | LCEL (pipe chains) | LangGraph |
|-----------|-------------------|----------|
| State management | Manual — thread state through yourself | Automatic — graph merges node dicts |
| Branching | Awkward (use `RunnableBranch`) | First-class (conditional edges) |
| Inspection | Hard (callback handlers) | Built-in (stream events, checkpointing) |
| Best for | Linear chains | Multi-step agents, loops, branching |

In [ ]:
# ─── 5-A: Assemble and run the full two-node pipeline ─────────────────────────
from langgraph.graph import END, START, StateGraph

graph = StateGraph(AgentState)
graph.add_node("parser", parser_node)
graph.add_node("generator", generator_node)

graph.add_edge(START, "parser")
graph.add_edge("parser", "generator")
graph.add_edge("generator", END)

app = graph.compile()

print("Graph compiled. Running: START → parser → generator → END\n")

result = app.invoke({"parsed_content": None, "ideas": None})

print(f"\n=== PIPELINE RESULT: {len(result['ideas'])} ideas ===")
for i, item in enumerate(result["ideas"], 1):
    idea = item.idea
    print(f"\n{i}. {idea.title} [{idea.effort_level} effort]")
    print(f"   Description: {idea.description}")
    print(f"   Tools:       {', '.join(idea.target_tools)}")
    print(f"   Value:       {idea.value_proposition}")
    if item.next_steps:
        print(f"   First step:  {item.next_steps[0]}")

In [ ]:
# ─── 5-B: Stream events to observe state at each node boundary ───────────────
# .stream() yields one event dict per node completion.
# This is invaluable for debugging — you see exactly what each node
# returns before the next node starts.

print("Streaming graph execution:\n")

for step, event in enumerate(app.stream({"parsed_content": None, "ideas": None}), 1):
    for node_name, output in event.items():
        print(f"Step {step} — node: {node_name!r}")
        if node_name == "parser":
            pc = output.get("parsed_content")
            if pc:
                print(f"  parsed_content.summary: {pc.summary}")
        elif node_name == "generator":
            ideas = output.get("ideas", [])
            print(f"  ideas count: {len(ideas)}")
            for idea_out in ideas:
                print(f"    - {idea_out.idea.title} [{idea_out.idea.effort_level}]")
        print()

---

## Part 6 — Scoring Extension: Adding a Third Pipeline Node

---

The two-node pipeline produces ideas, but does not help you decide which to build first. A **scorer node** adds a numeric priority score to each idea based on configurable weights.

### Scoring Formula

```
effort_score = {"low": 3, "medium": 2, "high": 1}[idea.effort_level]

# value_score: proxy using word count of value_proposition
# In production, replace this with a second LLM call that rates 1-5
value_score = min(len(value_proposition.split()) / 10, 3.0)

priority = 0.6 * effort_score + 0.4 * value_score
```

Higher priority = more impactful AND easier to build. Adjust the weights to match your priorities.

In [ ]:
# ─── 6-A: Scorer node and extended state ─────────────────────────────────────

class ScoredIdeaOutput(BaseModel):
    """An IdeaOutput with an added priority score."""
    idea_output: IdeaOutput
    priority_score: float
    score_reasoning: str


class ExtendedState(AgentState, total=False):
    """AgentState extended with the scored_ideas key for the scorer node."""
    scored_ideas: list[ScoredIdeaOutput] | None


EFFORT_SCORE = {"low": 3, "medium": 2, "high": 1}


def scorer_node(state: ExtendedState) -> dict:
    """Score and rank ideas by effort (lower = better) and value proposition richness."""
    ideas = state.get("ideas") or []
    scored = []

    for item in ideas:
        effort_score = EFFORT_SCORE.get(item.idea.effort_level.lower(), 2)
        value_words = len(item.idea.value_proposition.split())
        value_score = min(value_words / 10, 3.0)
        priority = round(0.6 * effort_score + 0.4 * value_score, 2)

        scored.append(
            ScoredIdeaOutput(
                idea_output=item,
                priority_score=priority,
                score_reasoning=(
                    f"effort={item.idea.effort_level} ({effort_score}/3), "
                    f"value_words={value_words}"
                ),
            )
        )

    scored.sort(key=lambda x: -x.priority_score)
    print(f"  [scorer] ranked {len(scored)} ideas")
    return {"scored_ideas": scored}


# Build extended graph: parser → generator → scorer
graph_ext = StateGraph(ExtendedState)
graph_ext.add_node("parser", parser_node)
graph_ext.add_node("generator", generator_node)
graph_ext.add_node("scorer", scorer_node)

graph_ext.add_edge(START, "parser")
graph_ext.add_edge("parser", "generator")
graph_ext.add_edge("generator", "scorer")
graph_ext.add_edge("scorer", END)

app_ext = graph_ext.compile()
print("Extended pipeline: START → parser → generator → scorer → END")

In [ ]:
# ─── 6-B: Run the extended pipeline and display ranked results ────────────────
print("Running extended pipeline...\n")

result_ext = app_ext.invoke(
    {"parsed_content": None, "ideas": None, "scored_ideas": None}
)

print(f"\n=== RANKED IDEAS (highest priority first) ===")
for rank, scored in enumerate(result_ext["scored_ideas"], 1):
    idea = scored.idea_output.idea
    print(f"\n#{rank} [score={scored.priority_score:.2f}] {idea.title}")
    print(f"   Effort: {idea.effort_level}  |  Tools: {', '.join(idea.target_tools)}")
    print(f"   Value:  {idea.value_proposition}")
    print(f"   Score breakdown: {scored.score_reasoning}")

---

## Part 7 — Debugging: Inspecting State at Each Step

---

### Common Pipeline Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| **JSON parse failure** | `ideas` is empty | LLM returned prose, not JSON | Improve prompt; rely on `extract_json_array` fallback |
| **Missing fields** | `KeyError` in `IdeaOutput()` | LLM omitted required keys | Include full schema in system prompt |
| **Wrong effort_level** | `"moderate"` instead of `"medium"` | LLM ignored enum | Add `exactly one of: low / medium / high` |
| **Empty tool list** | `target_tools = []` | LLM hallucinated or skipped field | Check `d.get("target_tools", [])` default |
| **Stale state** | Generator sees old `parsed_content` | State not cleared between runs | Always pass `{"parsed_content": None, "ideas": None}` |

### Debugging Checklist

1. Test each node in isolation before wiring into the graph
2. Print the raw LLM response before JSON extraction
3. Test `extract_json_array()` with a copy of the raw response
4. Use `.stream()` to see what each node returns
5. Validate every `IdeaOutput` — Pydantic will surface missing fields immediately

In [ ]:
# ─── 7-A: Debug mode — print raw LLM response before extraction ──────────────

def generator_node_debug(state: AgentState) -> dict:
    """Generator node with verbose logging — use while debugging."""
    content = state["parsed_content"]
    tool_list = "\n".join(
        f"- {t.name} ({t.category}): {t.description}" for t in content.tools
    )

    resp = _llm.invoke(
        [
            SystemMessage(_SYSTEM_PROMPT),
            HumanMessage(f"Available free tools:\n{tool_list}\n\nGenerate 3-5 ideas:"),
        ]
    )

    print("=== RAW LLM RESPONSE (first 600 chars) ===")
    print(resp.content[:600])
    if len(resp.content) > 600:
        print(f"... [{len(resp.content) - 600} more chars]")
    print("" + "=" * 45 + "\n")

    ideas_data = extract_json_array(resp.content)
    print(f"Extracted {len(ideas_data)} raw idea dicts")

    ideas = [
        IdeaOutput(
            idea=IdeaConcept(
                title=d["title"],
                description=d.get("description", ""),
                target_tools=d.get("target_tools", []),
                value_proposition=d.get("value_proposition", ""),
                effort_level=d.get("effort_level", "medium"),
            ),
            reasoning=d.get("reasoning", ""),
            next_steps=d.get("next_steps", []),
        )
        for d in ideas_data
        if isinstance(d, dict) and "title" in d
    ]
    return {"ideas": ideas}


debug_graph = StateGraph(AgentState)
debug_graph.add_node("parser", parser_node)
debug_graph.add_node("generator", generator_node_debug)
debug_graph.add_edge(START, "parser")
debug_graph.add_edge("parser", "generator")
debug_graph.add_edge("generator", END)

debug_app = debug_graph.compile()
debug_result = debug_app.invoke({"parsed_content": None, "ideas": None})
print(f"\nDebug run produced {len(debug_result['ideas'])} ideas")

---

## Part 8 ★ — Advanced Techniques (Bonus)

---

### Conditional Routing

LangGraph supports conditional edges — useful for retrying on failure or branching between scorer variants:

```python
def route_after_generator(state: AgentState) -> str:
    if not state.get("ideas"):
        return "retry_generator"   # re-run with a stronger prompt
    return "scorer"

graph.add_conditional_edges("generator", route_after_generator)
```

### Async Invocation

For production use (web server, batch jobs), run the graph asynchronously:

```python
result = await app.ainvoke({"parsed_content": None, "ideas": None})
```

All LangChain and LangGraph components support `ainvoke` and `astream` — no other changes needed.

### Checkpointing for Long Pipelines

For pipelines with expensive steps, add a checkpointer so a crash mid-run can resume from the last successful node:

```python
from langgraph.checkpoint.sqlite import SqliteSaver

with SqliteSaver.from_conn_string(":memory:") as memory:
    app = graph.compile(checkpointer=memory)
    result = app.invoke(
        {"parsed_content": None, "ideas": None},
        config={"configurable": {"thread_id": "run-001"}},
    )
```

### `with_structured_output` — Skip Manual JSON Extraction

If you are on LangChain 0.2+ with a model that supports tool calling, you can bypass `extract_json_array` entirely:

```python
from pydantic import BaseModel

class IdeasResponse(BaseModel):
    ideas: list[IdeaConcept]

structured_llm = _llm.with_structured_output(IdeasResponse)
response = structured_llm.invoke([SystemMessage(prompt), HumanMessage(tools_text)])
# response.ideas is already a list[IdeaConcept] — no JSON parsing needed
```

See **Example 13 — Structured Output** for a deep dive on this pattern.

---

## Exercises

---

### Exercise 1 — Change the Domain

Replace `FREE_TOOLS_RAW` with tools from a domain you care about (marketing tools, design tools, data platforms, no-code tools). Re-run the full pipeline.

**Goal:** confirm the generator produces domain-appropriate ideas without any other changes.

**Challenge:** add a `url` field to `FreeTool` and include it in the formatted prompt. Does the LLM reference URLs in `next_steps`?

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
MY_TOOLS_RAW = [
    # TODO: replace with tools from your domain
    # Format: {"name": "...", "category": "...", "description": "..."}
    {"name": "TODO: Tool 1", "category": "Category A", "description": "What it does"},
    {"name": "TODO: Tool 2", "category": "Category B", "description": "What it does"},
    {"name": "TODO: Tool 3", "category": "Category A", "description": "What it does"},
]

# TODO: define a custom parser_node that uses MY_TOOLS_RAW instead of FREE_TOOLS_RAW
# TODO: wire it into a new graph and run app.invoke({"parsed_content": None, "ideas": None})

### Exercise 2 — Harden JSON Extraction

Modify `extract_json_array` to also handle the case where the LLM returns a single object `{...}` instead of an array `[{...}]`. Wrap it in a list automatically rather than returning empty.

**Stretch goal:** add a retry — if `extract_json_array` returns empty, call the LLM a second time with `"Return only a raw JSON array, no other text."` appended.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────
def extract_json_array_hardened(raw: str) -> list:
    """TODO: extend extract_json_array to handle single-object responses."""
    result = extract_json_array(raw)
    if result:
        return result

    # TODO: also try to parse a single JSON object and wrap it in a list
    # Hint: re.search(r"(?s)(\{.*\})", raw) then json.loads() the match
    return []


# Test it
single_obj = '{"title": "Single Idea", "effort_level": "low"}'
print(f"Hardened extraction of single object: {extract_json_array_hardened(single_obj)}")

### Exercise 3 — LLM-Powered Scorer

Replace the heuristic scorer in Part 6 with an LLM-based scorer. Ask the model to rate each idea 1-5 for **feasibility** and **market potential**, then compute a weighted score.

**Hint:** send one LLM call per idea with a prompt like:
```
Rate this idea on feasibility (1-5) and market_potential (1-5).
Return JSON only: {"feasibility": N, "market_potential": N}
```

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────
def scorer_node_llm(state) -> dict:
    """TODO: replace heuristic scoring with LLM-based scoring."""
    ideas = state.get("ideas") or []
    scored = []

    for item in ideas:
        # TODO: call _llm with a prompt that asks for feasibility + market_potential
        # TODO: extract the JSON response using extract_json_array or json.loads()
        # TODO: compute priority = 0.5 * feasibility + 0.5 * market_potential
        # TODO: append ScoredIdeaOutput to scored
        pass

    scored.sort(key=lambda x: -x.priority_score)
    return {"scored_ideas": scored}


print("Implement scorer_node_llm above, then wire it into app_ext instead of scorer_node.")

### Exercise 4 — Fetch Live Data

The production `src/tools.py` fetches the free-for-dev catalogue from GitHub at runtime and parses the Markdown. Replace the inline `FREE_TOOLS_RAW` with a live fetch.

The source URL:
```
https://raw.githubusercontent.com/ripienaar/free-for-dev/master/README.md
```

**Steps:**
1. `requests.get(url)` → markdown text
2. `re.findall(r"##\s*(.*?)\s*\n(.*?)(?=\n##|$)", content, re.DOTALL)` → sections
3. For each section, `re.findall(r"[\*\-]\s*\[([^\]]+)\]\([^)]*\)\s*-\s*(.*?)(?=\n|$)", ...)` → tool lines
4. Build `FreeTool` objects, pass to a custom `parser_node`, run the pipeline

In [ ]:
# ── Exercise 4 Starter ────────────────────────────────────────────────────────
FREE_FOR_DEV_URL = "https://raw.githubusercontent.com/ripienaar/free-for-dev/master/README.md"


def fetch_free_tools(url: str = FREE_FOR_DEV_URL, limit: int = 30) -> list[dict]:
    """TODO: fetch and parse the free-for-dev README.
    Return a list of dicts with keys: name, category, description.
    """
    import requests
    # TODO: requests.get(url, timeout=10)
    # TODO: parse sections and tool lines using the regex patterns above
    # TODO: return list of dicts (cap at `limit` tools)
    return []


# Uncomment to test once implemented:
# live_tools = fetch_free_tools(limit=20)
# print(f"Fetched {len(live_tools)} tools")
# for t in live_tools[:5]:
#     print(f"  [{t['category']}] {t['name']}: {t['description'][:60]}")
print("Implement fetch_free_tools() above, then run the pipeline with live data.")

---

## What's Next?

You now have a working two-agent ideation pipeline with typed state, robust JSON extraction, and an optional scorer node. Here's where to go deeper:

### Same pattern, different domain
- **Example 3 — Prospect Agent** ([`../3-prospect-agent/`](../3-prospect-agent/)): the same parser → generator pattern applied to sales research — parser extracts company data, generator writes personalised outreach.

### Deeper structured output
- **Example 13 — Structured Output** ([`../13-structured-output/`](../13-structured-output/)): goes deeper on `with_structured_output()`, Pydantic v2 validators, and complex nested schemas — no manual JSON extraction needed.

### Plan-then-execute
- **Example 22 — Plan Execute** ([`../22-plan-execute/`](../22-plan-execute/)): extends the pipeline pattern to multi-step execution — the planner generates a task list, sub-agents execute each task in sequence or in parallel.

### Further reading
- Wei, J. et al. (2022). *Chain-of-Thought Prompting.* NeurIPS 2022. https://arxiv.org/abs/2201.11903
- Brown, T. et al. (2020). *GPT-3: Few-Shot Learners.* NeurIPS 2020. https://arxiv.org/abs/2005.14165
- LangGraph conceptual guide: https://langchain-ai.github.io/langgraph/concepts/
- LangGraph how-to — subgraphs and multi-agent: https://langchain-ai.github.io/langgraph/how-tos/

---

*Workshop complete. The next step is Example 13 — go deeper on structured extraction.*

---

## Exercise Answer Key

Use this section after attempting the exercises yourself. These are reference implementations — not the only correct approach.

---

### Exercise 1 Answer — Change the Domain

Swap `FREE_TOOLS_RAW` inside `parser_node` for your own list. The only requirement is that each dict has `name`, `category`, and `description` keys to match `FreeTool`. Everything downstream — the generator prompt, JSON extraction, scoring — is completely domain-agnostic.

**What good output looks like:** ideas should reference tool names from your list exactly (not hallucinated ones) and the combination logic should be plausible (e.g., Canva + Mailchimp + Zapier = automated visual newsletter pipeline).

**If ideas feel generic:** add more descriptive `description` fields — the LLM uses those descriptions heavily when deciding what combinations make sense.

In [ ]:
# Sample answer for Exercise 1 — design/creative tools domain
DESIGN_TOOLS_RAW = [
    {"name": "Figma",     "category": "Design",   "description": "Collaborative UI design, free starter tier"},
    {"name": "Canva",     "category": "Design",   "description": "Drag-and-drop graphic design, free tier"},
    {"name": "Unsplash",  "category": "Assets",   "description": "Free high-resolution stock photos API"},
    {"name": "Iconify",   "category": "Assets",   "description": "Open-source icon library, 150k+ icons"},
    {"name": "Coolors",   "category": "Color",    "description": "Color palette generator, free API"},
    {"name": "Vercel",    "category": "Hosting",  "description": "Free frontend hosting with instant previews"},
]

# Validate that all tools parse cleanly before running the pipeline
design_tools = [FreeTool(**t) for t in DESIGN_TOOLS_RAW]
print(f"Design tools validated: {len(design_tools)} tools")
for t in design_tools:
    print(f"  [{t.category}] {t.name}: {t.description}")

### Exercise 2 Answer — Harden JSON Extraction

The key addition is a fourth fallback that tries to parse a single `{...}` object and wraps it in a list. This handles the surprisingly common case where an LLM returns one idea instead of an array.

**Retry pattern:** on the second call, prepend a `HumanMessage` with `"Return only a raw JSON array with no prose."` — one retry is almost always sufficient.

In [ ]:
# Sample answer for Exercise 2
def extract_json_array_hardened_answer(raw: str) -> list:
    text = raw.strip()

    # Try 1: direct parse (clean JSON array or object)
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            return [data]
    except json.JSONDecodeError:
        pass

    # Try 2: strip ```json fence (array or object)
    fence = re.search(r"```(?:json)?\s*([\[\{].*?[\]\}])\s*```", text, re.DOTALL)
    if fence:
        try:
            data = json.loads(fence.group(1))
            return data if isinstance(data, list) else [data]
        except json.JSONDecodeError:
            pass

    # Try 3: find any [...] or {...} block inside prose
    for pattern in [r"(?s)(\[.*\])", r"(?s)(\{.*\})"]:
        match = re.search(pattern, text)
        if match:
            try:
                data = json.loads(match.group(1))
                return data if isinstance(data, list) else [data]
            except json.JSONDecodeError:
                continue

    return []


# Verify all cases
answer_cases = [
    ('[{"title": "A"}]',               "array"),
    ('{"title": "B"}',                 "single object"),
    ('```json\n{"title": "C"}\n```',   "fenced single object"),
    ('No JSON here.',                   "no JSON → []"),
]
for raw, label in answer_cases:
    result = extract_json_array_hardened_answer(raw)
    print(f"{label:30} → {result}")

### Exercise 4 Answer — Fetch Live Data

The production `src/tools.py` uses `requests` plus two regex patterns. The `limit` parameter caps tool count to avoid flooding the LLM context window.

**What to expect from live data:** the free-for-dev list has 300+ tools across 40+ categories. Filtering to 20-30 high-quality tools usually produces better ideas than sending all 300, because the generator can reason more carefully over a focused catalogue.

In [ ]:
# Sample answer for Exercise 4
def fetch_free_tools_answer(url: str = FREE_FOR_DEV_URL, limit: int = 25) -> list[dict]:
    """Fetch and parse the free-for-dev README into a list of tool dicts."""
    import requests
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except Exception as e:
        print(f"Fetch failed: {e}")
        return []

    content = response.text
    sections = re.findall(r"##\s*(.*?)\s*\n(.*?)(?=\n##|$)", content, re.DOTALL)

    tools = []
    for section_title, section_body in sections:
        section_title = section_title.strip()
        if not section_title or "Table of Contents" in section_title:
            continue
        tool_lines = re.findall(
            r"[\*\-]\s*\[([^\]]+)\]\([^)]*\)\s*[-\u2013]\s*(.*?)(?=\n|$)",
            section_body,
        )
        for name, description in tool_lines:
            tools.append({
                "name": name.strip(),
                "category": section_title,
                "description": re.sub(r"\s+", " ", description.strip()),
            })
            if len(tools) >= limit:
                return tools

    return tools


# Uncomment to run (requires internet access):
# live_tools = fetch_free_tools_answer(limit=20)
# print(f"Fetched {len(live_tools)} tools")
# for t in live_tools[:5]:
#     print(f"  [{t['category']}] {t['name']}: {t['description'][:60]}")
print("fetch_free_tools_answer defined. Uncomment above to run with internet access.")